# Milestone 3: SME Lending Policy Simulation and Risk-Based Pricing

This notebook uses the probability of default (PD) from the Milestone 2
logistic regression scorecard to simulate three lending decisions:
auto-approve, refer for manual review, and auto-decline.

We will select the PD cutoffs on the development sample, then apply them
to the validation sample and compare the proposed policy with the simulated
historical booking decisions from Milestone 1. We will also estimate
break-even lending rates for historically booked loans.

## 1. Prepare the Validation Dataset

### 1.1 Locate the Input Files

- `validation_scored.csv`: scores, predicted PDs, and synthetic default
  outcomes for the 4,500 validation applicants.
- `sme_booking_decision.csv`: simulated historical booking decisions for
  all 15,000 applicants, including applicants who were not booked.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Works whether Jupyter starts in the repo root or in notebooks/.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

validation_path = root / "reports" / "sme_scorecard_v1" / "validation_scored.csv"
booking_files = list((root / "data").rglob("sme_booking_decision.csv"))

if not validation_path.is_file():
    raise FileNotFoundError(
        f"Could not find {validation_path}. Run the export cell in Notebook 2."
    )

if len(booking_files) != 1:
    raise ValueError(
        f"Expected one booking file under {root / 'data'}, found {len(booking_files)}."
    )

booking_path = booking_files[0]

print("Validation scores:", validation_path.relative_to(root))
print("Booking decisions:", booking_path.relative_to(root))

Validation scores: reports/sme_scorecard_v1/validation_scored.csv
Booking decisions: data/raw/sme_booking_decision.csv


### 1.2 Load the Files and Check Applicant IDs

Both files should contain exactly one row per applicant. Before joining
them, we check that `applicant_id` is present and unique in each file.
We also inspect the row counts and column names.

In [2]:
validation_scores = pd.read_csv(validation_path, dtype={"applicant_id": "string"})
booking_history = pd.read_csv(booking_path, dtype={"applicant_id": "string"})

for name, frame in [
    ("Validation scores", validation_scores),
    ("Booking history", booking_history),
]:
    ids = frame["applicant_id"]
    assert ids.notna().all(), f"{name} has missing applicant IDs."
    assert ids.is_unique, f"{name} has duplicate applicant IDs."

    print(f"{name}: {len(frame):,} rows")
    print("Columns:", frame.columns.tolist())

Validation scores: 4,500 rows
Columns: ['applicant_id', 'predicted_pd', 'exact_score', 'display_score', 'default_12m']
Booking history: 15,000 rows
Columns: ['applicant_id', 'booked']


### 1.3 Join Validation Scores to Historical Booking Decisions

Join the files on `applicant_id`, keeping all 4,500 validation applicants.
The join should match each applicant to exactly one historical booking record.

The synthetic default outcome will be used to evaluate the proposed policy,
including for applicants who were not historically booked. It will not be
used to make an individual lending decision.

In [3]:
policy_validation = validation_scores.merge(
    booking_history,
    on="applicant_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

matched = policy_validation["_merge"].eq("both")
print(f"Matched validation applicants: {matched.sum():,} / {len(matched):,}")

assert len(policy_validation) == 4_500, "Unexpected validation sample size."
assert matched.all(), "Some validation applicants have no booking record."

policy_validation = policy_validation.drop(columns="_merge")
display(policy_validation.head())

Matched validation applicants: 4,500 / 4,500


,applicant_id,predicted_pd,exact_score,display_score,default_12m,booked
0,SME009631,0.005972,752.846595,753,0,0
1,SME004779,0.274761,453.917714,454,1,0
2,SME013797,0.705175,320.997525,321,1,0
3,SME012704,0.548798,369.778611,370,1,0
4,SME007272,0.094709,546.743910,547,0,1


## 2. Establish the Historical Booking Baseline

Before testing a new lending policy, we measure how the simulated historical
policy performed on the 4,500 validation applicants.

For booked and not-booked applicants, we report the number of applicants,
their share of the validation sample, their 12-month default rate, and
their average scorecard PD.

The booked group's applicant share is the historical booking rate. Its
default rate is the baseline we will compare with the proposed policy.
We can also report outcomes for the not-booked group because this dataset
is synthetic.

In [4]:
for column in ["booked", "default_12m"]:
    assert policy_validation[column].isin([0, 1]).all(), (
        f"{column} must contain only 0 and 1."
    )

assert policy_validation["predicted_pd"].between(0, 1).all(), (
    "Predicted PDs must be between 0 and 1 with no missing values."
)

historical_baseline = (
    policy_validation
    .groupby("booked", as_index=False)
    .agg(
        applicants=("applicant_id", "size"),
        defaults=("default_12m", "sum"),
        default_rate=("default_12m", "mean"),
        average_pd=("predicted_pd", "mean"),
    )
    .sort_values("booked", ascending=False)
)

historical_baseline["historical_status"] = historical_baseline["booked"].map(
    {1: "Booked", 0: "Not booked"}
)
historical_baseline["applicant_share"] = (
    historical_baseline["applicants"] / len(policy_validation)
)

display(
    historical_baseline.set_index("historical_status")[
        ["applicants", "applicant_share", "defaults", "default_rate", "average_pd"]
    ].style.format({
        "applicants": "{:,.0f}",
        "applicant_share": "{:.2%}",
        "defaults": "{:,.0f}",
        "default_rate": "{:.2%}",
        "average_pd": "{:.2%}",
    })
)

,applicants,applicant_share,defaults,default_rate,average_pd
historical_status,,,,,
Booked,"2,974",66.09%,424,14.26%,14.97%
Not booked,"1,526",33.91%,612,40.10%,37.40%


## Export Development Scores for Milestone 3

Save one row per development applicant with their applicant ID, predicted
12-month probability of default, and synthetic default outcome.

Milestone 3 will use these predictions to compare candidate approval and
decline cutoffs. After selecting the cutoffs, we will apply them to the
separate validation sample to evaluate the proposed policy.

## 3. Load the Development Predictions

Load the 10,500 predictions saved by Notebook 2. We will use this sample
to choose the lending cutoffs and the separate validation sample to test them.

In [5]:
development_path = validation_path.with_name("development_scored.csv")
development_scores = pd.read_csv(development_path, dtype={"applicant_id": "string"})

print(f"Loaded {len(development_scores):,} development predictions.")

Loaded 10,500 development predictions.


## 4. Compare Candidate PD Cutoffs

We test three policies that auto-approve roughly the lowest-risk 60%, 65%,
or 70% of development applicants. The 65% scenario is close to the
historical booking volume and gives us a starting point for comparison.

All three policies use the same upper cutoff: the 85th percentile of
development PDs. Applicants at or above this cutoff are auto-declined.
Applicants at or below the lower cutoff are auto-approved. Everyone
between the two cutoffs is referred for manual review.

We compare the share auto-approved, the default rate among auto-approved
applicants, and the share sent for review. These are candidate policies;
we will evaluate the selected policy on the separate validation sample.

In [6]:
dev_pd = development_scores["predicted_pd"]
dev_default = development_scores["default_12m"]

assert dev_pd.between(0, 1).all(), "Missing or invalid development PDs."
assert dev_default.isin([0, 1]).all(), "Invalid default labels."

decline_cutoff = dev_pd.quantile(0.85)
scenarios = []

for target_share in [0.60, 0.65, 0.70]:
    approve_cutoff = dev_pd.quantile(target_share)
    assert approve_cutoff < decline_cutoff, "The cutoffs must be distinct."

    approved = dev_pd <= approve_cutoff
    declined = dev_pd >= decline_cutoff
    referred = ~(approved | declined)

    scenarios.append({
        "scenario": f"{target_share:.0%} target",
        "approve_cutoff": approve_cutoff,
        "decline_cutoff": decline_cutoff,
        "auto_approve_rate": approved.mean(),
        "approved_default_rate": dev_default[approved].mean(),
        "manual_review_rate": referred.mean(),
        "auto_decline_rate": declined.mean(),
    })

policy_scenarios = pd.DataFrame(scenarios).set_index("scenario")
display(policy_scenarios.style.format("{:.2%}"))

,approve_cutoff,decline_cutoff,auto_approve_rate,approved_default_rate,manual_review_rate,auto_decline_rate
scenario,,,,,,
60% target,21.18%,47.40%,60.00%,8.86%,25.00%,15.00%
65% target,24.68%,47.40%,65.00%,9.95%,20.00%,15.00%
70% target,28.97%,47.40%,70.00%,10.99%,15.00%,15.00%


## 5. Apply the Selected Policy to Validation

We selected the 65% scenario because its development auto-approval rate
was closest to the historical booking rate among the scenarios tested.

We apply the exact development cutoffs to the 4,500 validation applicants.
The cutoffs display as 24.68% for auto-approval and 47.40% for auto-decline.
Applicants between the two go to manual review.

We compare the auto-approved group with the historically booked group
on the same validation sample. Referrals are reported separately because
their final lending decisions are unknown.

In [7]:
# Read the exact cutoffs from the selected development scenario.
selected_policy = policy_scenarios.loc["65% target"]
approval_pd = selected_policy["approve_cutoff"]
decline_pd = selected_policy["decline_cutoff"]

val_pd = policy_validation["predicted_pd"]
assert val_pd.between(0, 1).all(), "Missing or invalid validation PDs."

policy_validation["proposed_decision"] = "Manual review"
policy_validation.loc[val_pd <= approval_pd, "proposed_decision"] = "Auto-approve"
policy_validation.loc[val_pd >= decline_pd, "proposed_decision"] = "Auto-decline"

groups = {
    "Historical: booked": policy_validation["booked"].eq(1),
    "Proposed: auto-approve": policy_validation["proposed_decision"].eq("Auto-approve"),
    "Proposed: manual review": policy_validation["proposed_decision"].eq("Manual review"),
    "Proposed: auto-decline": policy_validation["proposed_decision"].eq("Auto-decline"),
}

validation_comparison = pd.DataFrame([
    {
        "group": name,
        "applicants": mask.sum(),
        "share_of_applications": mask.mean(),
        "defaults": policy_validation.loc[mask, "default_12m"].sum(),
        "default_rate": policy_validation.loc[mask, "default_12m"].mean(),
    }
    for name, mask in groups.items()
]).set_index("group")

display(validation_comparison.style.format({
    "applicants": "{:,.0f}",
    "share_of_applications": "{:.2%}",
    "defaults": "{:,.0f}",
    "default_rate": "{:.2%}",
}))

,applicants,share_of_applications,defaults,default_rate
group,,,,
Historical: booked,"2,974",66.09%,424,14.26%
Proposed: auto-approve,"2,956",65.69%,288,9.74%
Proposed: manual review,877,19.49%,308,35.12%
Proposed: auto-decline,667,14.82%,440,65.97%


## 6. Compare Who Would Be Approved

Cross-tabulate historical booking status and proposed decisions on the
validation sample. For each combination, show the applicant count and
the simulated 12-month default rate.

This shows how many historically unbooked applicants the new policy
would auto-approve, and where historically booked applicants would go
under the proposed policy.

In [8]:
policy_switches = (
    policy_validation
    .assign(historical_status=policy_validation["booked"].map({
        1: "Booked",
        0: "Not booked",
    }))
    .groupby(["historical_status", "proposed_decision"])
    .agg(
        applicants=("applicant_id", "size"),
        default_rate=("default_12m", "mean"),
    )
)

display(policy_switches.style.format({
    "applicants": "{:,.0f}",
    "default_rate": "{:.2%}",
}))

## 7. Estimate Break-Even Loan Pricing

For each historically booked validation loan, estimate the annual interest
rate needed to cover funding costs, operating costs, and expected credit loss.

This is a simplified one-year simulation. We assume a constant loan balance,
8% funding cost, 2% operating cost, and 45% loss given default (LGD).
These are illustrative assumptions. We exclude fees and taxes.

Expected credit loss as a share of exposure = predicted 12-month PD × LGD.

The break-even rate covers the assumed costs. We also calculate a target
rate by adding a 2-percentage-point pre-tax margin on the loan balance.

In [9]:
funding_cost = 0.08
operating_cost = 0.02
lgd = 0.45
target_margin = 0.02

booked_pricing = policy_validation.loc[
    policy_validation["booked"].eq(1),
    ["applicant_id", "predicted_pd"],
].copy()

booked_pricing["expected_loss_rate"] = booked_pricing["predicted_pd"] * lgd
booked_pricing["break_even_rate"] = (
    funding_cost + operating_cost + booked_pricing["expected_loss_rate"]
)
booked_pricing["target_rate"] = booked_pricing["break_even_rate"] + target_margin

print(f"Historically booked loans priced: {len(booked_pricing):,}")
print(f"Median break-even rate: {booked_pricing['break_even_rate'].median():.2%}")

display(booked_pricing.head().style.format({
    "predicted_pd": "{:.2%}",
    "expected_loss_rate": "{:.2%}",
    "break_even_rate": "{:.2%}",
    "target_rate": "{:.2%}",
}))

Historically booked loans priced: 2,974
Median break-even rate: 14.48%


,applicant_id,predicted_pd,expected_loss_rate,break_even_rate,target_rate
4,SME007272,9.47%,4.26%,14.26%,16.26%
6,SME000702,2.52%,1.13%,11.13%,13.13%
7,SME000670,15.99%,7.19%,17.19%,19.19%
9,SME004061,8.51%,3.83%,13.83%,15.83%
11,SME013017,9.51%,4.28%,14.28%,16.28%


## 8. Demo: Look Up an Applicant

Enter an applicant ID to see the scorecard score, predicted default
probability, historical booking status, and proposed lending decision.

Estimated pricing is shown only for historically booked loans. An applicant
can be auto-approved under the proposed policy and still have no pricing
result in this demo because they were not historically booked.

In [10]:
def show_applicant(applicant_id):
    applicant = policy_validation.loc[
        policy_validation["applicant_id"].eq(applicant_id),
        ["applicant_id", "display_score", "predicted_pd", "booked",
         "proposed_decision"],
    ].copy()

    if applicant.empty:
        print(f"{applicant_id} was not found in the validation sample.")
        return

    applicant["historical_status"] = applicant.pop("booked").map({
        1: "Booked",
        0: "Not booked",
    })

    applicant = applicant.merge(
        booked_pricing[["applicant_id", "break_even_rate", "target_rate"]],
        on="applicant_id",
        how="left",
        validate="one_to_one",
    )

    display(applicant.style.format({
        "predicted_pd": "{:.2%}",
        "break_even_rate": "{:.2%}",
        "target_rate": "{:.2%}",
    }, na_rep="Not priced").hide(axis="index"))


show_applicant("SME007272")  # Historically booked: pricing is shown
show_applicant("SME009631")  # Not historically booked: pricing is blank

applicant_id,display_score,predicted_pd,proposed_decision,historical_status,break_even_rate,target_rate
SME007272,547,9.47%,Auto-approve,Booked,14.26%,16.26%


applicant_id,display_score,predicted_pd,proposed_decision,historical_status,break_even_rate,target_rate
SME009631,753,0.60%,Auto-approve,Not booked,Not priced,Not priced


## 9. Export Policy and Pricing Results for the Dashboard

Save the application-policy decisions, illustrative pricing, and exact
selected PD cutoffs for use in the integrated dashboard.

These results cover the 4,500 application-validation applicants.
Historical booking status remains separate from the proposed decision.

Default outcomes are excluded from the dashboard decision exports.

In [11]:
from pathlib import Path
import json

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

policy_report_directory = root / "reports" / "sme_lending_policy_v1"
policy_report_directory.mkdir(parents=True, exist_ok=True)

# Retrieve the exact cutoffs from our selected development scenario.
selected_policy = policy_scenarios.loc["65% target"]

policy_config = {
    "policy_version": "sme_lending_policy_v1",
    "selected_scenario": "65% target",
    "approve_cutoff": float(selected_policy["approve_cutoff"]),
    "decline_cutoff": float(selected_policy["decline_cutoff"]),
    "cutoff_source": "Development-sample PD distribution",
    "application_validation_accounts": int(len(policy_validation)),
    "pricing_assumptions": {
        "lgd": 0.45,
        "funding_cost_rate": 0.08,
        "operating_cost_rate": 0.02,
        "target_margin": 0.02
    },
    "pricing_scope": "Historically booked application-validation accounts",
    "pricing_note": "Illustrative rates, not historical quoted rates"
}

policy_decisions_export = policy_validation.drop(
    columns=["default_12m"],
    errors="ignore"
).copy()

pricing_export = booked_pricing.drop(
    columns=["default_12m"],
    errors="ignore"
).copy()

for name, frame in [
    ("policy_decisions", policy_decisions_export),
    ("booked_pricing", pricing_export)
]:
    assert frame["applicant_id"].notna().all()
    assert frame["applicant_id"].is_unique

    frame.to_csv(
        policy_report_directory / f"{name}.csv",
        index=False
    )

config_path = policy_report_directory / "policy_config.json"
config_path.write_text(
    json.dumps(policy_config, indent=2),
    encoding="utf-8"
)

# Verify the exported decision count and saved cutoffs.
reloaded_decisions = pd.read_csv(
    policy_report_directory / "policy_decisions.csv",
    dtype={"applicant_id": "string"}
)
reloaded_config = json.loads(config_path.read_text(encoding="utf-8"))

assert reloaded_decisions["applicant_id"].tolist() == (
    policy_decisions_export["applicant_id"].tolist()
)
assert reloaded_config == policy_config

print(f"Application-policy decisions saved: {len(reloaded_decisions):,}")
print(f"Historically booked pricing rows saved: {len(pricing_export):,}")
print(f"Exact approval PD cutoff: {policy_config['approve_cutoff']:.4%}")
print(f"Exact decline PD cutoff: {policy_config['decline_cutoff']:.4%}")
print("Policy export and reload checks passed.")

Application-policy decisions saved: 4,500
Historically booked pricing rows saved: 2,974
Exact approval PD cutoff: 24.6795%
Exact decline PD cutoff: 47.3969%
Policy export and reload checks passed.
